In [19]:
TEGENPLOEG = 'C:\\Users\\StijnHuysman\\OneDrive - mateco cloud\\GITHUB REPOS\\HAANTJES\\TEGENPLOEG.HTML'

# functions

In [23]:
def fetch_player_ALL(SPELERID):
    url = "https://app.basketballstatsvlaanderen.be/players/" + SPELERID + "season=2425"
    url = 'https://app.basketballstatsvlaanderen.be/players/' + SPELERID
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    # Extract subfields from 'GameTeam' if present
    for game in games_data:
        if 'GameTeam' in game and isinstance(game['GameTeam'], dict):
            game['GameTeam_TEAM'] = game['GameTeam'].get('Name', {})
            game['GameTeam_GameResult'] = game['GameTeam'].get('Game', {}).get('Result')
            game['GameTeam_GameGuid'] = game['GameTeam'].get('Game', {}).get('Guid')
            game['GameTeam_GameDate'] = game['GameTeam'].get('Game', {}).get('Date')
            game['GameTeam_AwayTeamName'] = game['GameTeam'].get('Game', {}).get('AwayTeam', {}).get('Name')
            game['GameTeam_HomeTeamName'] = game['GameTeam'].get('Game', {}).get('HomeTeam', {}).get('Name')

            # Add AGE column by trimming after the 2nd last space in GameTeam_TEAM
            for game in games_data:
                team_name = game.get('GameTeam_TEAM', '')
                if isinstance(team_name, str):
                    parts = team_name.split(' ')
                    if len(parts) > 2:
                        game['AGE'] = ' '.join(parts[-2:])
                    else:
                        game['AGE'] = team_name
                else:
                    game['AGE'] = None
    

    # filtered_games_data = [
    #     {k: game[k] for k in fields_to_keep if k in game}
    #     for game in games_data
    # ]

    df_games = pd.DataFrame(games_data)
    
    df_games = df_games.drop(columns=['id', 'GameTeamId', 'Stints', 'createdAt', 'updatedAt', 'GameTeam'])
    return df_games


In [25]:
fetch_player_ALL('BVBL748910')


,Guid,Name,Number,FunctionLetter,Starter,TotalMinutes,NormalizedMinutes,FreeThrows,FieldGoals,ThreePointers,...,Plus,Minus,Badges,GameTeam_TEAM,GameTeam_GameResult,GameTeam_GameGuid,GameTeam_GameDate,GameTeam_AwayTeamName,GameTeam_HomeTeamName,AGE
0,BVBL748910,Milan De Brabander,4,S,True,28,32,2,8,3,...,65,-55,[plusminus],BBC Haantjes Certifisc Oudenaarde HSE B,83- 77,BVBL25269130OVHSE31BFC,2025-09-26T20:30:00.000Z,BBC Haantjes Certifisc Oudenaarde HSE B,BBC Olympia Denderleeuw HSE B,HSE B
1,BVBL748910,Milan De Brabander,14,S,True,22,25,7,20,0,...,82,-41,"[1p, 2p, p, offence, plusminus]",BBC Haantjes Certifisc Oudenaarde J18 A,104- 56,BVBL25261037OR00040508,2025-09-07T13:00:00.000Z,KBBC Wezen-Vrienden Geraardsbergen J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,J18 A
2,BVBL748910,Milan De Brabander,14,S,True,25,29,0,6,3,...,27,-39,[2p],BBC Haantjes Certifisc Oudenaarde HSE B,43- 66,BVBL25269130OVHSE31BCA,2025-09-13T20:15:00.000Z,BBC Hotshots Destelbergen HSE B,BBC Haantjes Certifisc Oudenaarde HSE B,HSE B
3,BVBL748910,Milan De Brabander,14,S,True,27,33,2,20,0,...,66,-36,"[1p, 2p, p, offence, plusminus]",BBC Haantjes Certifisc Oudenaarde J18 A,74- 61,BVBL25269170INJ1821BCA,2025-09-13T15:30:00.000Z,Koninklijke BBC Oostkamp J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,J18 A
4,BVBL748910,Milan De Brabander,14,S,True,22,26,4,18,0,...,59,-51,"[1p, 2p, p, offence]",BBC Haantjes Certifisc Oudenaarde J18 A,74- 64,BVBL25261037OR00040507,2025-08-30T19:00:00.000Z,BBC Olympia Denderleeuw J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,J18 A
5,BVBL748910,Milan De Brabander,14,S,True,29,33,0,14,0,...,66,-55,[2p],BBC Haantjes Certifisc Oudenaarde J18 A,83- 67,BVBL25269100BLAJ18PNAC,2025-09-06T17:00:00.000Z,Phantoms Basket Boom J18 A,BBC Haantjes Certifisc Oudenaarde J18 A,J18 A
6,BVBL748910,Milan De Brabander,14,S,True,28,34,0,16,3,...,62,-46,"[2p, 3p, p]",BBC Haantjes Certifisc Oudenaarde HSE B,56- 73,BVBL25269130BOVHSEVR03,2025-08-31T11:00:00.000Z,BBC Haantjes Certifisc Oudenaarde HSE B,BBC Hotshots Destelbergen HSE A,HSE B
7,BVBL748910,Milan De Brabander,14,S,True,21,24,4,10,0,...,39,-27,[1p],BBC Haantjes Certifisc Oudenaarde HSE B,81- 44,BVBL25261037OR00040505,2025-09-06T19:30:00.000Z,Blue Rocks Ronse-Kluisbergen HSE B,BBC Haantjes Certifisc Oudenaarde HSE B,HSE B
8,BVBL748910,Milan De Brabander,14,S,True,26,29,5,10,6,...,55,-36,"[1p, 3p, p]",BBC Haantjes Certifisc Oudenaarde J18 A,53- 73,BVBL25269170INJ1821BDC,2025-09-20T13:00:00.000Z,BBC Haantjes Certifisc Oudenaarde J18 A,Wytewa Roeselare J18 A,J18 A
9,BVBL748910,Milan De Brabander,15,S,True,18,18,1,2,0,...,30,-40,[],BBC Haantjes Certifisc Oudenaarde HSE B,73- 52,BVBL25269130OVHSE31BMC,2025-09-20T18:30:00.000Z,BBC Haantjes Certifisc Oudenaarde HSE B,BC Lede HSE B,HSE B


In [26]:
def FETCH_PLAYERS_AVG(SPELERID):
    df = fetch_player_ALL(SPELERID)
    # Unnest AGE from GameTeam column if not already present
    if 'AGE' not in df.columns:
        df['AGE'] = df['GameTeam'].apply(lambda x: ' '.join(x['Name'].split(' ')[-2:]) if isinstance(x, dict) and 'Name' in x else None)

    grouped = df.groupby(['Name', 'AGE'])
    result = grouped.agg({
        'TotalMinutes': 'mean',
        'NormalizedMinutes': 'mean',
        'FreeThrows': 'mean',
        'FieldGoals': 'mean',
        'ThreePointers': 'mean',
        'TotalScore': ['min', 'mean', 'median', 'max' , 'count'],
        'PlusMinus': 'mean',
        'Faults': 'mean',
        'Plus': 'mean',
        'Minus': 'mean',
       
    })

    # Flatten MultiIndex columns
    result.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in result.columns.values]
    # Rename and rearrange columns to match the desired order and names
    result = result.rename(columns={
         'TotalScore_count': 'WEDSTRIJDEN',
        
        'TotalMinutes_mean': 'Avg TotalMinutes',
        'NormalizedMinutes_mean': 'Avg NormalizedMinutes',
        'FreeThrows_mean': 'Avg FreeThrows',
        'FieldGoals_mean': 'Avg FieldGoals',
        'ThreePointers_mean': 'Avg ThreePointers',
        'TotalScore_min': 'Min TotalScore',
        'TotalScore_mean': 'Avg TotalScore',
        'TotalScore_median': 'Median TotalScore',
        'TotalScore_max': 'Max TotalScore',
       'Faults_mean': 'Avg Faults',
        'PlusMinus_mean': 'Avg PlusMinus',
        
        'Plus_mean': 'Avg Plus',
        'Minus_mean': 'Avg Minus'
    })

    # Reorder columns to match the desired template
    desired_order = [
        'WEDSTRIJDEN',
        'Avg TotalMinutes',
        'Avg NormalizedMinutes',
        'Avg FreeThrows',
        'Avg FieldGoals',
        'Avg ThreePointers',
        'Min TotalScore',
        'Avg TotalScore',
        'Median TotalScore',
        'Max TotalScore',
        
        'Avg PlusMinus',
        'Avg Faults',
        'Avg Plus',
        'Avg Minus'
    ]
    # Keep index columns (Name, AGE) at the front
    result = result.reset_index()[['Name', 'AGE'] + desired_order]
    # Round all columns containing 'Avg' to 1 decimal place
    avg_cols = [col for col in result.columns if 'Avg' in col]
    result[avg_cols] = result[avg_cols].round(1)
    return result




In [27]:
import pandas as pd

# Extract player data directly from soup (tbody)
tbody = soup.find('tbody')
rows = tbody.find_all('tr') if tbody else []

player_list = []
for row in rows:
    cols = row.find_all('td')
    if len(cols) >= 2:
        # Extract jersey number (remove icon if present)
        jersey_text = cols[0].get_text(strip=True)
        jersey_number = pd.to_numeric(jersey_text.split()[-1], errors='coerce')
        name = cols[1].get_text(strip=True)
        if pd.notna(jersey_number) and name:
            player_list.append({'jersey_number': int(jersey_number), 'name': name})

player_data = pd.DataFrame(player_list)
player_data

,jersey_number,name
0,4,Axl Houtteman
1,5,Arno Vergucht
2,6,Vic Van Maele
3,7,Mathis Vanstreels
4,8,Sam Devisscher
5,8,Thomas Cathenis
6,9,Rien Verspille
7,10,Yannes Rabaey
8,11,Stan Nulens
9,11,Alejandro Cambero Collado


In [28]:
import json
import pandas as pd

# Load the ALLPLAYERS.json file
with open(r'C:\Users\StijnHuysman\OneDrive - mateco cloud\GITHUB REPOS\HAANTJES\ALLPLAYERS.json', 'r', encoding='utf-8') as f:
    allplayers_data = json.load(f)

# Convert to DataFrame and expand 'rows' if necessary
allplayers_df = pd.DataFrame(allplayers_data)
if isinstance(allplayers_df['rows'].iloc[0], dict):
    rows_expanded = allplayers_df['rows'].apply(pd.Series)
    allplayers_df = pd.concat([allplayers_df.drop(columns=['rows']), rows_expanded], axis=1)

# Example: find Guid, LidNr, Name for a given player name
def find_player_info(player_name):
    match = allplayers_df[allplayers_df['Name'] == player_name]
    if not match.empty:
        return match[['Guid', 'LidNr', 'Name']].iloc[0].to_dict()
    else:
        return None


player_data['Guid'] = player_data['name'].map(lambda n: find_player_info(n)['Guid'] if find_player_info(n) else None)
player_data['LidNr'] = player_data['name'].map(lambda n: find_player_info(n)['LidNr'] if find_player_info(n) else None)
# print(player_data)

In [29]:
player_data

,jersey_number,name,Guid,LidNr
0,4,Axl Houtteman,BVBL716294,716294
1,5,Arno Vergucht,BVBL751340,751340
2,6,Vic Van Maele,BVBL744284,744284
3,7,Mathis Vanstreels,BVBL711490,711490
4,8,Sam Devisscher,BVBL750843,750843
5,8,Thomas Cathenis,BVBL715031,715031
6,9,Rien Verspille,BVBL715077,715077
7,10,Yannes Rabaey,BVBL750225,750225
8,11,Stan Nulens,BVBL735879,735879
9,11,Alejandro Cambero Collado,BVBL770757,770757


In [30]:
# Add columns from FETCH_PLAYERS_AVG function to player_data DataFrame
for index, row in player_data.iterrows():
    guid = row['Guid']
    if pd.notna(guid):
        try:
            # Get player stats using the FETCH_PLAYERS_AVG function
            player_stats = FETCH_PLAYERS_AVG(guid)
            
            if not player_stats.empty:
                # Get the first row of stats (assuming one player per Guid)
                stats_row = player_stats.iloc[0]
                
                # Add all stats columns to player_data
                for col in player_stats.columns:
                    if col not in ['Name', 'AGE']:  # Skip duplicate columns
                        player_data.loc[index, col] = stats_row[col]
        except Exception as e:
            print(f"Error fetching stats for {row['name']} (Guid: {guid}): {e}")

print(f"Updated player_data with stats for {len(player_data)} players")
player_data

Updated player_data with stats for 15 players


,jersey_number,name,Guid,LidNr,WEDSTRIJDEN,Avg TotalMinutes,Avg NormalizedMinutes,Avg FreeThrows,Avg FieldGoals,Avg ThreePointers,Min TotalScore,Avg TotalScore,Median TotalScore,Max TotalScore,Avg PlusMinus,Avg Faults,Avg Plus,Avg Minus
0,4,Axl Houtteman,BVBL716294,716294,8.0,14.8,17.9,1.4,2.8,0.0,1.0,4.1,3.5,8.0,-17.5,2.2,19.5,-37.0
1,5,Arno Vergucht,BVBL751340,751340,3.0,23.0,27.0,1.7,15.3,3.0,14.0,20.0,18.0,28.0,7.0,3.3,41.3,-34.3
2,6,Vic Van Maele,BVBL744284,744284,9.0,12.3,14.8,1.3,0.7,0.3,0.0,2.3,2.0,7.0,-16.1,2.6,14.4,-30.6
3,7,Mathis Vanstreels,BVBL711490,711490,8.0,14.0,16.9,0.1,1.2,0.0,0.0,1.4,1.0,4.0,-15.1,2.2,16.1,-31.2
4,8,Sam Devisscher,BVBL750843,750843,3.0,15.3,17.7,0.3,2.0,0.0,1.0,2.3,2.0,4.0,-23.3,1.7,13.7,-37.0
5,8,Thomas Cathenis,BVBL715031,715031,6.0,15.7,18.3,1.7,1.7,0.0,0.0,3.3,3.5,7.0,-16.2,0.8,15.2,-31.3
6,9,Rien Verspille,BVBL715077,715077,8.0,11.9,13.9,0.1,5.0,0.0,2.0,5.1,4.0,10.0,-13.0,1.5,16.4,-29.4
7,10,Yannes Rabaey,BVBL750225,750225,8.0,13.8,17.4,0.2,2.2,0.0,0.0,2.5,2.0,9.0,-18.1,2.1,15.5,-33.6
8,11,Stan Nulens,BVBL735879,735879,7.0,17.1,21.0,0.3,2.6,0.0,0.0,2.9,2.0,10.0,-18.3,0.6,19.0,-37.3
9,11,Alejandro Cambero Collado,BVBL770757,770757,7.0,13.3,15.3,0.1,0.3,0.0,0.0,0.4,0.0,2.0,-21.3,0.6,10.1,-31.4


In [31]:
player_data.to_excel('player_stats.xlsx', index=False)

## FUNCTIONS
Let's extract player data from the identified tbody elements: